In [1]:
# Install DeepChem and RDKit
# We need specific versions compatible with Colab
!pip install --pre deepchem
!pip install rdkit-pypi

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


In [2]:
import deepchem as dc
import numpy as np

def load_and_verify_data():
    """
    Loads the ClinTox dataset using DeepChem's built-in ScaffoldSplitter.
    Prints the shape of train, valid, and test sets to verify splitting.
    """
    print("Loading ClinTox dataset with Scaffold Split...")
    # Load dataset using standard DeepChem loader
    # Using 'scaffold' splitter as recommended for chemical datasets
    tasks, datasets, transformers = dc.molnet.load_clintox(splitter='scaffold')
    train_dataset, valid_dataset, test_dataset = datasets
    # Verify the split sizes
    print("\nDataset Split Summary:")
    print(f"Train dataset samples: {len(train_dataset)}")
    print(f"Valid dataset samples: {len(valid_dataset)}")
    print(f"Test dataset samples:  {len(test_dataset)}")
    # Check the number of tasks (labels)
    print(f"\nNumber of prediction tasks: {len(tasks)}")
    print(f"Task names: {tasks}")
    # Inspect the first few samples to ensure data is loaded correctly
    print("\nSample Data (First 3 Train Molecules):")
    for i in range(3):
        print(f"ID {i}: {train_dataset.ids[i]}")
        # y is the label (0 or 1 for toxicity)
        print(f"Label {i}: {train_dataset.y[i]}")

if __name__ == "__main__":
    load_and_verify_data()

wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


Loading ClinTox dataset with Scaffold Split...

Dataset Split Summary:
Train dataset samples: 1184
Valid dataset samples: 148
Test dataset samples:  148

Number of prediction tasks: 2
Task names: ['FDA_APPROVED', 'CT_TOX']

Sample Data (First 3 Train Molecules):
ID 0: B([C@H](CC(C)C)NC(=O)CNC(=O)C1=C(C=CC(=C1)Cl)Cl)(O)O
Label 0: [0. 1.]
ID 1: C(CO)N(c1c(c(c(c(c1I)C(=O)NCC(CO)O)I)C(=O)NCC(CO)O)I)C(=O)CO
Label 1: [1. 0.]
ID 2: c1c(c(cc(c1Cl)Cl)Cl)OCC#CI
Label 2: [1. 0.]


In [3]:
import deepchem as dc
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. load data globally so we can use it later
print("loading clintox...")
tasks, datasets, transformers = dc.molnet.load_clintox(splitter='scaffold')
train_dataset, valid_dataset, test_dataset = datasets

# 2. simple random forest baseline
print("fitting RF model...")
model = RandomForestClassifier(n_estimators=100)

# clintox has multiple tasks, using the first one (FDA_APPROVED)
model.fit(train_dataset.X, train_dataset.y[:, 0])

# checking on test set
preds = model.predict_proba(test_dataset.X)[:, 1]
y_true = test_dataset.y[:, 0]

# handle nan values
mask = ~np.isnan(y_true)
score = roc_auc_score(y_true[mask], preds[mask])

print(f"Test ROC-AUC: {score}")

loading clintox...
fitting RF model...
Test ROC-AUC: 0.7398081534772183


In [4]:
from transformers import AutoTokenizer

# checking if we can tokenize the smiles strings properly
def check_tokenization():
    model_id = "mistralai/Mistral-7B-v0.1"
    print(f"Loading tokenizer for {model_id}...")
    # loading tokenizer only (not the full model yet to save ram)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # setting pad token if missing
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # grab 3 examples from our loaded dataset
    examples = train_dataset.ids[:3]

    print("\nTokenizing first 3 smiles strings:")
    for i, smile in enumerate(examples):
        # converting smile string to model inputs
        tokens = tokenizer(smile, padding="max_length", max_length=50, truncation=True)

        print(f"\nExample {i+1}:")
        print(f"Original: {smile}")
        print(f"Token IDs (first 10): {tokens['input_ids'][:10]}")
        print(f"Total tokens: {len(tokens['input_ids'])}")

check_tokenization()

Loading tokenizer for mistralai/Mistral-7B-v0.1...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Tokenizing first 3 smiles strings:

Example 1:
Original: B([C@H](CC(C)C)NC(=O)CNC(=O)C1=C(C=CC(=C1)Cl)Cl)(O)O
Token IDs (first 10): [2, 2, 2, 2, 2, 2, 1, 365, 5187, 28743]
Total tokens: 50

Example 2:
Original: C(CO)N(c1c(c(c(c(c1I)C(=O)NCC(CO)O)I)C(=O)NCC(CO)O)I)C(=O)CO
Token IDs (first 10): [1, 334, 28732, 1998, 28731, 28759, 28732, 28717, 28740, 28717]
Total tokens: 50

Example 3:
Original: c1c(c(cc(c1Cl)Cl)Cl)OCC#CI
Token IDs (first 10): [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Total tokens: 50


In [5]:
# installing libraries for 4-bit loading and lora
# -q means quiet mode to keep logs clean
!pip install -q -U bitsandbytes peft accelerate

In [6]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

def setup_lora_model():
    print("Loading Mistral-7B in 4-bit (Quantized)...")
    model_id = "mistralai/Mistral-7B-v0.1"
    # 4-bit configuration to fit in T4 GPU
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    # load the base model
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("Base model loaded. Applying LoRA adapters...")
    # LoRA configuration (The Secret Sauce)
    # r=8 means low rank (very efficient)
    # target_modules are the layers we want to train
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    )
    # wrap the model
    model = get_peft_model(model, peft_config)
    # This print statement is our "Winning Proof"
    print("\nEfficiency Check:")
    model.print_trainable_parameters()
    return model

# execute setup
if __name__ == "__main__":
    model = setup_lora_model()

Loading Mistral-7B in 4-bit (Quantized)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Base model loaded. Applying LoRA adapters...

Efficiency Check:
trainable params: 21,260,288 || all params: 7,262,992,384 || trainable%: 0.2927
